# Extract the *Leopardus tilcayo* mitogenome from WGS reads (SRX30196846)

Runs entirely inside this free Colab VM — no dependency on Galaxy's infrastructure (which has a broken GetOrganelle container image as of today).

**How to use:** Run cells top to bottom with Shift+Enter. Cell 1 will trigger an automatic kernel restart (Colab does this itself for conda setup) — after it restarts, just continue running from Cell 2 onward.

Budget ~30-90 minutes total depending on read depth and Colab's assigned CPU/RAM for the session.

In [ ]:
# 1) Bootstrap conda in Colab (this cell restarts the kernel automatically - that's expected)
!pip install -q condacolab
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


**After the restart above, continue from here.**

In [ ]:
# 2) Install the tools we need
!mamba install -y -c bioconda -c conda-forge sra-tools getorganelle fastp seqkit

[+] 0.0s
[+] 0.0s
[+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch     1%
bioconda/linux-64     ⣾  
bioconda/noarch        1%[+] 0.2s
conda-forge/linux-64   2%
conda-forge/noarch     3%
bioconda/linux-64      3%
bioconda/noarch       19%[+] 0.3s
conda-forge/linux-64   3%
conda-forge/noarch     7%
bioconda/linux-64     19%
bioconda/noarch       28%[+] 0.4s
conda-forge/linux-64   6%
conda-forge/noarch    11%
bioconda/linux-64     40%
bioconda/noarch       53%[+] 0.5s
conda-forge/linux-64   8%
conda-forge/noarch    15%
bioconda/linux-64     52%
bioconda/noarch       66%[+] 0.6s
conda-forge/linux-64   9%
conda-forge/noarch    17%
bioconda/linux-64     69%
bioconda/noarch       84%[+] 0.7s
conda-forge/linux-64  10%
conda-forge/noarch    20%
bioconda/linux-64     84%
bioconda/noarch      100%bioconda/noarch                                   
[+] 0.8s
conda-forge/linux-64  11%
conda-forge/noarch    22%
bioconda/linux-64     91%[+] 0.9s
conda-forge/linux-64  11%
conda-forge/noarch    23%


In [ ]:
!pip install -q pysradb
!pysradb --version

pysradb 2.5.1


In [ ]:
# Resolve the experiment accession to its actual run accession(s)
!pysradb srx-to-srr SRX30196846

experiment_accession	run_accession	study_accession	study_title	experiment_title	experiment_desc	organism_taxid	organism_name	library_name	library_strategy	library_source	library_selection	library_layout	sample_accession	sample_title	biosample	bioproject	instrument	instrument_model	instrument_model_desc	total_spots	total_size	run_total_spots	run_total_bases
SRX30196846	SRR35083483	SRP540473	WGS for tiger cat species complex	WGS short reads	WGS short reads	3431183	Leopardus tilcayo	MNHN-L2	WGS	GENOMIC	RANDOM	PAIRED	SRS26256705		SAMN48723862	PRJNA1176476	Illumina NovaSeq 6000	Illumina NovaSeq 6000	ILLUMINA	121206194	12238709577	121206194	36604270588


In [ ]:
# 3) Download a SUBSAMPLE of the WGS reads
import os
os.makedirs('work/fastq', exist_ok=True)
%cd work

N_SPOTS = 20_000_000
!fastq-dump -X {N_SPOTS} --split-files --gzip -O fastq/ SRR35083483   # <- paste the resolved SRR here

!ls -lh fastq/
!df -h /content

/content/work/work
Read 20000000 spots for SRR35083483
Written 20000000 spots for SRR35083483
total 2.9G
-rw-r--r-- 1 root root 1.4G Sep 24 14:46 SRR35083483_1.fastq.gz
-rw-r--r-- 1 root root 1.5G Sep 24 14:46 SRR35083483_2.fastq.gz
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   27G  200G  12% /


In [ ]:
# 4) Adapter-trim only (GetOrganelle's own docs recommend adapter-trimmed reads
#    WITHOUT aggressive quality filtering, so quality filtering is disabled here)
!fastp -i fastq/*_1.fastq.gz -I fastq/*_2.fastq.gz \
    -o trimmed_1.fastq.gz -O trimmed_2.fastq.gz \
    --disable_quality_filtering --thread 4 \
    --json fastp.json --html fastp.html

# free the raw copy now that trimmed reads exist
!rm -rf fastq/
!df -h /content

Read1 before filtering:
total reads: 20000000
total bases: 3020000000
Q20 bases: 2930427800(97.034%)
Q30 bases: 2803761101(92.8398%)
Q40 bases: 0(0%)

Read2 before filtering:
total reads: 20000000
total bases: 3020000000
Q20 bases: 2913489282(96.4732%)
Q30 bases: 2751831355(91.1202%)
Q40 bases: 0(0%)

Read1 after filtering:
total reads: 20000000
total bases: 2653943267
Q20 bases: 2592600206(97.6886%)
Q30 bases: 2493746580(93.9638%)
Q40 bases: 0(0%)

Read2 after filtering:
total reads: 20000000
total bases: 2653943267
Q20 bases: 2569981717(96.8363%)
Q30 bases: 2436004723(91.7881%)
Q40 bases: 0(0%)

Filtering result:
reads passed filter: 40000000
reads failed due to low quality: 0
reads failed due to too many N: 0
reads failed due to too short: 0
reads failed due to adapter dimer: 0
reads with adapter trimmed: 14262618
bases trimmed due to adapters: 732113466

Duplication rate: 5.09574%

Insert size peak (evaluated by paired-end reads): 179

JSON report: fastp.json
HTML report: fastp.htm

In [ ]:
# 5) Fetch GetOrganelle's built-in animal-mitochondria seed/label database
!get_organelle_config.py --add animal_mt


Python 3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]
DEPENDENCIES: Bowtie2 default; Blast 2.17.0
WORKING DIR: /content/work/work
/usr/local/bin/get_organelle_config.py --add animal_mt

Existing databases(s): 

Downloaded animal_mt.fasta (30285897 bytes), cost 6.82 s
bowtie2-build animal_mt.fasta ... finished
Downloaded animal_mt.fasta (6674918 bytes), cost 1.79 s
makeblastdb animal_mt.fasta ... finished

Total cost: 45.76 s


In [ ]:
# Confirm what's actually needed for this step still exists
!ls -lh trimmed_1.fastq.gz trimmed_2.fastq.gz

# 1) SRA download cache - no longer needed now that FASTQ is extracted
!rm -rf ~/ncbi

# 2) Conda/mamba's package download cache from the install step -
#    can reclaim a meaningful chunk, all safe to clear
!mamba clean --all -y

# 3) Leftover raw fastq/ folder, in case an earlier cell was rerun out
#    of order and the cleanup in cell 4 didn't fire
!rm -rf fastq/

# 4) Any go_output/ from a previous attempt - GetOrganelle can behave
#    oddly resuming into a directory with stale intermediate files from
#    a prior run, and SPAdes intermediates in there can be sizable
!rm -rf go_output/

!df -h /content

-rw-r--r-- 1 root root 8.5G Sep 24 09:44 trimmed_1.fastq.gz
-rw-r--r-- 1 root root 8.9G Sep 24 09:44 trimmed_2.fastq.gz
Collect information..
Cleaning index cache..
Cleaning lock files..
  Package file                                                      Size
──────────────────────────────────────────────────────────────────────────
  /usr/local/pkgs
──────────────────────────────────────────────────────────────────────────

  _openmp_mutex-4.5-20_gnu.conda                                    29kB
  archspec-0.2.5-pyhd8ed1ab_0.conda                                 51kB
  backports.zstd-1.5.0-py313h18e8e13_0.conda                       243kB
  blast-2.17.0-h66d330f_0.conda                                     85MB
  boltons-25.0.0-pyhd8ed1ab_0.conda                                302kB
  bowtie2-2.5.5-h63e9258_1.conda                                    11MB
  brotli-python-1.2.0-py313hf159716_1.conda                        368kB
  bzip2-1.0.8-hda65f42_9.conda                              

In [ ]:
# 6) Assemble the mitogenome
!get_organelle_from_reads.py \
    -1 trimmed_1.fastq.gz -2 trimmed_2.fastq.gz \
    -o go_output \
    -R 15 -k 21,45,65,85,105 \
    -F animal_mt -t 4


GetOrganelle v1.7.7.1

get_organelle_from_reads.py assembles organelle genomes from genome skimming data.
Find updates in https://github.com/Kinggerm/GetOrganelle and see README.md for more information.

Python 3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]
PLATFORM: Linux dcc5bdea9c91 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64
PYTHON LIBS: GetOrganelleLib 1.7.7.1; numpy 2.5.3; sympy 1.14.0; scipy 1.18.0
DEPENDENCIES: Bowtie2 default; SPAdes 4.3.0; Blast 2.17.0
GETORG_PATH=/root/.GetOrganelle
SEED  DB: animal_mt 0.0.1
LABEL DB: animal_mt 0.0.1
WORKING_DIR=/content/work/work
/usr/local/bin/get_organelle_from_reads.py -1 trimmed_1.fastq.gz -2 trimmed_2.fastq.gz -o go_output -R 15 -k 21,45,65,85,105 -F animal_mt -t 4

2026-09-24 15:43:08,548 - INFO: Pre-reading fastq ...
2026-09-24 15:43:08,548 - INFO: Estimating reads to use ... (to use all reads, set '--reduce-reads-for-coverage inf --max-reads inf')
2026-09-24 15:43:08,642 - INFO: Tastin

In [ ]:
# 7) Check the result and download it to your computer
import glob
!rm -f trimmed_1.fastq.gz trimmed_2.fastq.gz  # no longer needed once assembled
!ls -lh go_output/
!df -h /content
result = glob.glob('go_output/*complete*path_sequence*.fasta')
print('Complete assembly found:', result)

if result:
    !seqkit stats {result[0]}
    from google.colab import files
    files.download(result[0])
else:
    print('No fully circularized assembly - inspect go_output/ for scaffold/')
    print('*path_sequence*.fasta (incomplete) files instead, or the assembly graph.')

total 739M
-rw-r--r--  1 root root  17K Sep 24 18:37 animal_mt.K105.complete.graph1.1.path_sequence.fasta
-rw-r--r--  1 root root  18K Sep 24 18:37 animal_mt.K105.complete.graph1.selected_graph.gfa
-rw-r--r--  1 root root 307M Sep 24 17:39 extended_1_paired.fq
-rw-r--r--  1 root root  66M Sep 24 17:39 extended_1_unpaired.fq
-rw-r--r--  1 root root 307M Sep 24 17:39 extended_2_paired.fq
-rw-r--r--  1 root root  57M Sep 24 17:39 extended_2_unpaired.fq
-rw-r--r--  1 root root 4.5M Sep 24 18:37 extended_K105.assembly_graph.fastg
-rw-r--r--  1 root root  922 Sep 24 18:37 extended_K105.assembly_graph.fastg.extend-animal_mt.csv
-rw-r--r--  1 root root  43K Sep 24 18:37 extended_K105.assembly_graph.fastg.extend-animal_mt.fastg
drwxr-xr-x 11 root root 4.0K Sep 24 18:37 extended_spades
-rw-r--r--  1 root root 7.2K Sep 24 18:37 get_org.log.txt
drwxr-xr-x  2 root root 4.0K Sep 24 16:06 seed
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   25G  202G  11% /
Complete assembly f

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!df -h /content
!tail -50 go_output/get_org.log.txt

Filesystem      Size  Used Avail Use% Mounted on
overlay         226G  150G   77G  67% /
Find updates in https://github.com/Kinggerm/GetOrganelle and see README.md for more information.

Python 3.13.13 | packaged by conda-forge | (main, Apr  8 2026, 02:00:33) [GCC 14.3.0]
PLATFORM: Linux d38e47a5c839 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64
PYTHON LIBS: GetOrganelleLib 1.7.7.1; numpy 2.5.3; sympy 1.14.0; scipy 1.18.0
DEPENDENCIES: Bowtie2 default; SPAdes 4.3.0; Blast 2.17.0
GETORG_PATH=/root/.GetOrganelle
SEED  DB: animal_mt 0.0.1
LABEL DB: animal_mt 0.0.1
WORKING_DIR=/content/work
/usr/local/bin/get_organelle_from_reads.py -1 trimmed_1.fastq.gz -2 trimmed_2.fastq.gz -o go_output -R 15 -k 21,45,65,85,105 -F animal_mt -t 4

2026-09-23 11:48:30,935 - INFO: Pre-reading fastq ...
2026-09-23 11:48:30,935 - INFO: Estimating reads to use ... (to use all reads, set '--reduce-reads-for-coverage inf --max-reads inf')
2026-09-23 11:48:31,034 - INFO: Tasting 100000+100000 reads .

### Next steps
- Confirm length/circularity: a felid mitogenome should be ~16.5-17 kb, ideally one contig.
- Annotate with [MITOS2](http://mitos2.bioinf.uni-leipzig.de) (paste the FASTA in) to get gene boundaries, tRNAs, rRNAs.
- Compare/align against published *Leopardus tigrinus*-complex mitogenomes to sanity-check placement.